# Percentile thresholding & aggregated SDM outputs (prototype)

Notebook-only exploration: convert continuous **0–1** suitability rasters into binary layers and **potential** species-count surfaces (roost, in-flight, combined without double-counting per species).

**Not** observed richness or confirmed presence.

Helper functions are defined here so they can move into modules later.


## 1. Setup and configuration

Paths, outputs, and threshold labels are in the next code cell. Raster bands use `band_key` = `get_model_id([latin_name, activity_type])` (see §2).


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio as rio
from IPython.display import display
from rasterio.windows import Window

from sdm.utils import set_project_wd
from sdm.utils.io import load_pickled_model
from sdm.commands.modelling.utils import get_model_id
from sdm.models.core.pipeline_features import pipeline_selected_feature_names

set_project_wd(verbose=False)

REPO_ROOT = Path.cwd()
PATH_MODEL_RESULTS = REPO_ROOT / "data" / "sdm_models" / "model_results.csv"
PATH_TRAINING = REPO_ROOT / "data" / "sdm_models" / "training_data.parquet"
PATH_PREDICTIONS = REPO_ROOT / "data" / "sdm_predictions" / "all_predictions.tif"

OUT_ROOT = REPO_ROOT / "outputs" / "thresholding_prototype"
OUT_TABLES = OUT_ROOT / "tables"
OUT_RASTERS = OUT_ROOT / "rasters"
OUT_FIGS = OUT_ROOT / "figures"
for p in (OUT_TABLES, OUT_RASTERS, OUT_FIGS):
    p.mkdir(parents=True, exist_ok=True)

THRESHOLD_COLS = {
    "p00_min_presence": "threshold_p00_min_presence",
    "p10_main": "threshold_p10_main",
    "p25_conservative": "threshold_p25_conservative",
}

plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 120})
print("Output root:", OUT_ROOT)


## 2. Load and inspect source artefacts


In [ ]:
def load_model_results(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Missing model index: {path}")
    df = pd.read_csv(path)
    df["band_key"] = df.apply(
        lambda r: get_model_id([r["latin_name"], r["activity_type"]]), axis=1
    )
    return df


def load_training_data(path: Path) -> pd.DataFrame | gpd.GeoDataFrame:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing training parquet: {path}. Run `sdm train` (writes training_data.parquet)."
        )
    try:
        return gpd.read_parquet(path)
    except Exception:
        return pd.read_parquet(path)


def inspect_prediction_raster(path: Path) -> dict:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing prediction raster: {path}. Run `sdm predict` (writes all_predictions.tif)."
        )
    with rio.open(path) as src:
        return {
            "path": path,
            "count": src.count,
            "crs": src.crs,
            "transform": src.transform,
            "bounds": src.bounds,
            "shape": (src.height, src.width),
            "nodata": src.nodata,
            "descriptions": list(src.descriptions) if src.descriptions else [],
            "dtypes": [src.dtypes[i] for i in range(1, src.count + 1)],
        }


model_results = load_model_results(PATH_MODEL_RESULTS)
training_df = load_training_data(PATH_TRAINING)
pred_info = inspect_prediction_raster(PATH_PREDICTIONS)

display(model_results.head())
print("Training columns:", list(training_df.columns))
print("Training rows:", len(training_df))
if "class" in training_df.columns:
    print(
        "Presence:",
        int((training_df["class"] == 1).sum()),
        "Background:",
        int((training_df["class"] == 0).sum()),
    )
if "identifier" in training_df.columns:
    print("Unique training identifiers:", training_df["identifier"].nunique())
print("Raster shape:", pred_info["shape"], "bands:", pred_info["count"], "nodata:", pred_info["nodata"])
print("CRS:", pred_info["crs"])
print("Band descriptions:", pred_info["descriptions"])


### Identifier alignment

- `identifier` in CSV and training (e.g. `Nyctalus noctula_In flight`)
- `band_key` = `get_model_id([latin_name, activity_type])` matches **raster band descriptions** from `sdm predict` (`nyctalus_noctula_in_flight`)


In [ ]:
model_ids_from_results = set(model_results["identifier"])
model_ids_from_training = (
    set(training_df["identifier"].unique()) if "identifier" in training_df.columns else set()
)
model_ids_from_raster_bands = {d for d in pred_info["descriptions"] if d}
band_keys_from_results = set(model_results["band_key"])

in_results_and_training = model_ids_from_results & model_ids_from_training
band_keys_matched = sorted(band_keys_from_results & model_ids_from_raster_bands)

mr_use = (
    model_results[
        model_results["identifier"].isin(in_results_and_training)
        & model_results["band_key"].isin(band_keys_matched)
    ]
    .copy()
    .sort_values(["activity_type", "latin_name"])
    .reset_index(drop=True)
)

print(
    "Identifiers in both CSV and training:",
    len(in_results_and_training),
    "| band_key also in raster:",
    len(band_keys_matched),
)
print("CSV ids missing from training:", model_ids_from_results - model_ids_from_training)
print(
    "CSV band_key missing from raster:",
    sorted(band_keys_from_results - model_ids_from_raster_bands),
)
print("Training ids missing from CSV:", model_ids_from_training - model_ids_from_results)

if not band_keys_matched:
    raise RuntimeError("No raster bands match model_results; stopping.")

if (band_keys_from_results - model_ids_from_raster_bands) or (
    model_ids_from_results - model_ids_from_training
):
    print("\n*** WARNING: using intersection only; fix data or regenerate predictions. ***\n")

raster_band_keys_ordered = [
    d for d in pred_info["descriptions"] if d in band_keys_matched
]
print("Matched models (raster band order):", len(raster_band_keys_ordered))
mr_by_band = mr_use.set_index("band_key")

for d in raster_band_keys_ordered:
    if d not in mr_by_band.index:
        raise KeyError(f"Raster band description {d!r} has no row in model_results / training intersection")


## 3. Presence-based percentile thresholds


In [ ]:
# Retain raw presence scores for omission QA
score_arrays: dict[str, np.ndarray] = {}
rows = []
warnings_map: dict[str, str] = {}

for _, row in mr_use.iterrows():
    identifier = row["identifier"]
    band_key = row["band_key"]
    latin = row["latin_name"]
    act = row["activity_type"]

    sub_all = training_df[training_df["identifier"] == identifier]
    sub_pr = sub_all[sub_all["class"] == 1]
    n_presence = len(sub_pr)
    n_background = int((sub_all["class"] == 0).sum())

    warn_parts = []
    if n_presence < 30:
        warn_parts.append("n_presence<30")
    mcv = row.get("mean_cv_score")
    if pd.notna(mcv) and float(mcv) < 0.7:
        warn_parts.append("mean_cv<0.7")
    if pd.isna(mcv):
        warn_parts.append("mean_cv_missing")

    scores = np.array([], dtype=float)
    try:
        model = load_pickled_model(row["model_path"])
        X = sub_pr.drop(columns=["geometry"], errors="ignore").copy()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            scores = model.predict_proba(X)[:, 1]
    except Exception as e1:
        warn_parts.append(f"predict_fail:{e1!r}")
        try:
            model = load_pickled_model(row["model_path"])
            fn = pipeline_selected_feature_names(model)
            X = sub_pr[fn]
            scores = model.predict_proba(X)[:, 1]
        except Exception as e2:
            warn_parts.append(f"fallback_fail:{e2!r}")

    if scores.size:
        score_arrays[identifier] = scores
        pmin = float(scores.min())
        p10 = float(np.percentile(scores, 10))
        p25 = float(np.percentile(scores, 25))
        p50 = float(np.percentile(scores, 50))
        pmean = float(scores.mean())
        pmax = float(scores.max())
        t00, t10, t25 = pmin, p10, p25
    else:
        pmin = p10 = p25 = p50 = pmean = pmax = np.nan
        t00 = t10 = t25 = np.nan
        warn_parts.append("no_scores")

    rows.append(
        {
            "identifier": identifier,
            "band_key": band_key,
            "latin_name": latin,
            "activity_type": act,
            "n_presence": n_presence,
            "n_background": n_background,
            "mean_cv_score": row.get("mean_cv_score"),
            "std_cv_score": row.get("std_cv_score"),
            "presence_score_min": pmin,
            "presence_score_p10": p10,
            "presence_score_p25": p25,
            "presence_score_median": p50,
            "presence_score_mean": pmean,
            "presence_score_max": pmax,
            "threshold_p00_min_presence": t00,
            "threshold_p10_main": t10,
            "threshold_p25_conservative": t25,
            "warning": "; ".join(warn_parts),
        }
    )

threshold_summary = pd.DataFrame(rows).sort_values(["activity_type", "latin_name"])
path_thresh = OUT_TABLES / "threshold_summary.csv"
threshold_summary.to_csv(path_thresh, index=False)
print("Wrote", path_thresh)
display(threshold_summary)


## 4. Presence score distributions


In [ ]:
thresh_by_bk = threshold_summary.set_index("band_key")
nplots = len(raster_band_keys_ordered)
ncol = min(4, max(3, int(np.ceil(np.sqrt(nplots)))))
nrow = int(np.ceil(nplots / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 3.4, nrow * 2.6), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for i, bk in enumerate(raster_band_keys_ordered):
    ax = axes.flat[i]
    row = thresh_by_bk.loc[bk]
    ident = row["identifier"]
    sc = score_arrays.get(ident)
    ax.set_axis_on()
    if sc is None or sc.size == 0:
        ax.text(0.5, 0.5, "no scores", ha="center")
        continue
    ax.hist(sc, bins=min(40, max(10, len(sc) // 2)), density=True, color="#4C72B0", alpha=0.75)
    for x, ls in [(row["threshold_p00_min_presence"], ":"), (row["threshold_p10_main"], "-"), (row["threshold_p25_conservative"], "--")]:
        if pd.notna(x):
            ax.axvline(float(x), color="k", ls=ls, lw=1.5)
    mcv = row["mean_cv_score"]
    ttl = row["latin_name"] + chr(10) + f"{row['activity_type']} · n={int(row['n_presence'])}"
    if pd.notna(mcv):
        ttl += f" · CV={float(mcv):.2f}"
    ax.set_title(ttl, fontsize=8)
fig.suptitle("Presence-site predicted suitability with p00 (:), p10 (—), p25 (--)", fontsize=10)
plt.tight_layout()
fig.savefig(OUT_FIGS / "presence_score_distributions.png", bbox_inches="tight")
plt.show()

# Thresholds by species (p10) — one row per model
fig2, ax2 = plt.subplots(figsize=(10, max(4, 0.22 * len(threshold_summary))))
y = np.arange(len(threshold_summary))
ax2.barh(y, threshold_summary["threshold_p10_main"].fillna(0), color="#55A868")
ax2.set_yticks(y)
ax2.set_yticklabels(
    threshold_summary["latin_name"] + " — " + threshold_summary["activity_type"],
    fontsize=7,
)
ax2.set_xlabel("p10 threshold (presence scores)")
ax2.set_title("Per-model p10 threshold")
plt.tight_layout()
fig2.savefig(OUT_FIGS / "thresholds_by_species.png", bbox_inches="tight")
plt.show()


## 5. Apply thresholds to `all_predictions.tif` (windowed)


In [ ]:
th_lookup = threshold_summary.set_index("band_key")
BIN_NODATA = 255  # uint8 sentinel for invalid / continuous nodata
window_size = 256


def write_binary_raster(threshold_label: str, threshold_col: str) -> Path:
    out_path = OUT_RASTERS / f"binary_predictions_{threshold_label}.tif"
    with rio.open(PATH_PREDICTIONS) as src:
        prof = src.profile.copy()
        n = len(raster_band_keys_ordered)
        prof.update(count=n, dtype="uint8", nodata=BIN_NODATA)
        thresh_vec = []
        missing = []
        for bk in raster_band_keys_ordered:
            tval = th_lookup.loc[bk][threshold_col]
            if pd.isna(tval):
                missing.append(bk)
                thresh_vec.append(np.nan)
            else:
                thresh_vec.append(float(tval))
        if missing:
            print("WARN missing thresholds:", missing)

        heights = [(i, min(window_size, src.height - i)) for i in range(0, src.height, window_size)]
        widths = [(j, min(window_size, src.width - j)) for j in range(0, src.width, window_size)]
        with rio.open(out_path, "w", **prof) as dst:
            dst.descriptions = tuple(raster_band_keys_ordered)
            for r0, h in heights:
                for c0, w in widths:
                    win = Window(c0, r0, w, h)
                    block = src.read(window=win)
                    nod = src.nodata
                    outs = []
                    for jb, bk in enumerate(raster_band_keys_ordered):
                        ia = pred_info["descriptions"].index(bk)
                        contig = block[ia].astype(np.float64)
                        tval = thresh_vec[jb]
                        if np.isnan(tval):
                            outb = np.full(contig.shape, BIN_NODATA, dtype=np.uint8)
                            outs.append(outb)
                            continue
                        if nod is None or (isinstance(nod, float) and np.isnan(nod)):
                            valid = np.isfinite(contig)
                        else:
                            valid = np.isfinite(contig) & (contig != nod)
                        hi = np.zeros(contig.shape, dtype=np.uint8)
                        hi[valid] = (contig[valid] >= tval).astype(np.uint8)
                        hi[~valid] = BIN_NODATA
                        outs.append(hi)
                    dst.write(np.stack(outs), window=win)
        return out_path


for lab, col in THRESHOLD_COLS.items():
    write_binary_raster(lab, col)
    print("Binary stack:", OUT_RASTERS / f"binary_predictions_{lab}.tif")


## 6. Aggregate counts (roost / in-flight / all without double-count)


In [ ]:
# Map band index -> species for vectorised max-over-activity-per-species
latin_by_bk = threshold_summary.set_index("band_key")["latin_name"].to_dict()
act_by_bk = threshold_summary.set_index("band_key")["activity_type"].to_dict()
species_ordered = sorted({latin_by_bk[bk] for bk in raster_band_keys_ordered})

latin_to_idxs: dict[str, list[int]] = {s: [] for s in species_ordered}
for j, bk in enumerate(raster_band_keys_ordered):
    latin_to_idxs[latin_by_bk[bk]].append(j)
roost_idx = [j for j, bk in enumerate(raster_band_keys_ordered) if act_by_bk[bk] == "Roost"]
flight_idx = [j for j, bk in enumerate(raster_band_keys_ordered) if act_by_bk[bk] == "In flight"]


def summarize_binary_stack(
    binary_stack: np.ndarray, valid_band_mask: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    # binary_stack (nb, h, w); valid_band_mask same shape — valid pixel if any band not nodata
    nb, h, w = binary_stack.shape
    roost = (
        binary_stack[roost_idx].sum(axis=0).astype(np.uint16)
        if roost_idx
        else np.zeros((h, w), dtype=np.uint16)
    )
    inflight = (
        binary_stack[flight_idx].sum(axis=0).astype(np.uint16)
        if flight_idx
        else np.zeros((h, w), dtype=np.uint16)
    )
    acc = np.zeros((h, w), dtype=np.uint16)
    for sp, idxs in latin_to_idxs.items():
        sub = binary_stack[idxs]
        acc += sub.max(axis=0).astype(np.uint16)
    valid = valid_band_mask.any(axis=0)
    roost = np.where(valid, roost, 0)
    inflight = np.where(valid, inflight, 0)
    acc = np.where(valid, acc, 0)
    return valid, np.stack([acc, roost, inflight], axis=0)


def write_aggregate_for_label(threshold_label: str) -> None:
    bin_path = OUT_RASTERS / f"binary_predictions_{threshold_label}.tif"
    out_all = OUT_RASTERS / f"potential_species_count_all_{threshold_label}.tif"
    with rio.open(bin_path) as src:
        prof = src.profile.copy()
        prof.update(count=1, dtype="uint16", nodata=65535)
        prof.update(compress="deflate")
        h, w = src.height, src.width
        out_roost = OUT_RASTERS / f"potential_species_count_roost_{threshold_label}.tif"
        out_fl = OUT_RASTERS / f"potential_species_count_in_flight_{threshold_label}.tif"
        with rio.open(out_all, "w", **prof) as da, rio.open(out_roost, "w", **prof) as dr, rio.open(
            out_fl, "w", **prof
        ) as df:
            da.descriptions = ("potential_species_count_all",)
            dr.descriptions = ("potential_species_count_roost",)
            df.descriptions = ("potential_species_count_in_flight",)
            for r0 in range(0, h, window_size):
                for c0 in range(0, w, window_size):
                    win = Window(
                        c0,
                        r0,
                        min(window_size, w - c0),
                        min(window_size, h - r0),
                    )
                    b = src.read(window=win)
                    valid_b = b != BIN_NODATA
                    valid, agg3 = summarize_binary_stack(b, valid_b)
                    da.write(np.where(valid, agg3[0], 65535).astype(np.uint16), 1, window=win)
                    dr.write(np.where(valid, agg3[1], 65535).astype(np.uint16), 1, window=win)
                    df.write(np.where(valid, agg3[2], 65535).astype(np.uint16), 1, window=win)

    print("Wrote aggregates for", threshold_label)


for lab in THRESHOLD_COLS:
    write_aggregate_for_label(lab)


**Combined count rule:** for each species, `suitable = max(binary_roost, binary_in_flight)` (missing activity treated as 0), then sum over species. Roost / in-flight maps sum binary bands for that activity only (one model per species–activity).


## 7. QA summary (omission & suitable area)


In [ ]:
qa_rows = []
BIN_NODATA = 255

for _, tr in threshold_summary.iterrows():
    ident = tr["identifier"]
    bk = tr["band_key"]
    scores = score_arrays.get(ident)
    for tlab, tcol in THRESHOLD_COLS.items():
        thr = tr[tcol]
        if scores is not None and scores.size and pd.notna(thr):
            omit = float((scores < float(thr)).mean())
        else:
            omit = np.nan
        qa_rows.append(
            {
                "threshold_label": tlab,
                "identifier": ident,
                "band_key": bk,
                "latin_name": tr["latin_name"],
                "activity_type": tr["activity_type"],
                "threshold": thr,
                "n_presence": tr["n_presence"],
                "training_omission_rate": omit,
                "mean_cv_score": tr["mean_cv_score"],
                "warning": tr["warning"],
            }
        )

qa_df = pd.DataFrame(qa_rows)
# Raster cell counts — align assignments to qa_df row index
for tlab in THRESHOLD_COLS:
    bpath = OUT_RASTERS / f"binary_predictions_{tlab}.tif"
    sub = qa_df.loc[qa_df["threshold_label"] == tlab]
    ix_list = []
    suits_list = []
    valids_list = []
    with rio.open(bpath) as src:
        for idx, tr in sub.iterrows():
            bk = tr["band_key"]
            ib = raster_band_keys_ordered.index(bk) + 1
            band = src.read(ib).ravel()
            valid = band != BIN_NODATA
            suited = valid & (band == 1)
            ix_list.append(idx)
            suits_list.append(int(suited.sum()))
            valids_list.append(int(valid.sum()))
    qa_df.loc[ix_list, "suitable_cell_count"] = suits_list
    qa_df.loc[ix_list, "valid_cell_count"] = valids_list
    spct = np.where(np.array(valids_list) > 0, 100 * np.array(suits_list) / np.array(valids_list), np.nan)
    qa_df.loc[ix_list, "suitable_area_percent"] = spct

warn2 = []
for i, row in qa_df.iterrows():
    extras = []
    if pd.notna(row["threshold_label"]) and "p10_main" == row["threshold_label"]:
        eo = row["training_omission_rate"]
        if pd.notna(eo) and abs(eo - 0.10) > 0.08:
            extras.append("omission_not_near_10pct")
        if pd.notna(row["suitable_area_percent"]):
            if row["suitable_area_percent"] > 80:
                extras.append("suitable_pct>80")
            if row["suitable_area_percent"] < 1:
                extras.append("suitable_pct<1")
    if extras:
        warn2.append("; ".join(extras))
    else:
        warn2.append("")
qa_df["warning"] = qa_df["warning"].fillna("") + np.where(np.array(warn2) != "", "; " + np.array(warn2), "")

path_qa = OUT_TABLES / "threshold_qa_summary.csv"
qa_df.to_csv(path_qa, index=False)
print("Wrote", path_qa)
display(qa_df.head(20))


## 8. Map QA figures (downsampled)


In [ ]:
AGG_NODATA = 65535
STEP = 4  # increase for quicker plots on huge grids


def plot_count_map(geotiff_path: Path, outfile: Path, title: str) -> None:
    if not geotiff_path.is_file():
        print("skip missing", geotiff_path)
        return
    with rio.open(geotiff_path) as src:
        arr = src.read(1)[::STEP, ::STEP].astype(np.float64)
        arr = np.ma.masked_where(arr >= AGG_NODATA - 1, arr)
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(arr, cmap="viridis")
        plt.colorbar(im, ax=ax, shrink=0.7, label="Potential modelled species count")
        ax.set_title(title + chr(10) + "(note: suitability-based index, not observed richness)")
        ax.axis("off")
        plt.tight_layout()
        plt.savefig(outfile, bbox_inches="tight")
        plt.show()


plot_count_map(
    OUT_RASTERS / "potential_species_count_all_p10_main.tif",
    OUT_FIGS / "potential_species_count_all_p10_main.png",
    "Potential species count (all activities combined)",
)
plot_count_map(
    OUT_RASTERS / "potential_species_count_roost_p10_main.tif",
    OUT_FIGS / "potential_species_count_roost_p10_main.png",
    "Roost-only count (p10)",
)
plot_count_map(
    OUT_RASTERS / "potential_species_count_in_flight_p10_main.tif",
    OUT_FIGS / "potential_species_count_in_flight_p10_main.png",
    "In-flight-only count (p10)",
)

# Sensitivity triptych
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, lab, ttl in zip(
    axes,
    ["p00_min_presence", "p10_main", "p25_conservative"],
    ["p00 min presence", "p10 main", "p25 conservative"],
):
    pth = OUT_RASTERS / f"potential_species_count_all_{lab}.tif"
    if not pth.is_file():
        ax.axis("off")
        continue
    with rio.open(pth) as src:
        arr = src.read(1)[::STEP, ::STEP].astype(np.float64)
        arr = np.ma.masked_where(arr >= AGG_NODATA - 1, arr)
    im = ax.imshow(arr, cmap="viridis")
    ax.set_title(ttl)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Sensitivity: potential species count (all) across thresholds")
plt.tight_layout()
fig.savefig(OUT_FIGS / "potential_species_count_sensitivity_all.png", bbox_inches="tight")
plt.show()


## 9. Optional: contributions at one high-count pixel


In [ ]:
# Pick a pixel with high combined count under p10
p10_all = OUT_RASTERS / "potential_species_count_all_p10_main.tif"
if p10_all.is_file():
    with rio.open(p10_all) as src:
        z = src.read(1)
        vm = (z < AGG_NODATA - 1) & (z > 0)
        if vm.any():
            flat = np.argmax(z * vm)
            r, c = np.unravel_index(flat, z.shape)
            transform = src.transform
            x, y = rio.transform.xy(transform, r, c)
            print("Example cell (row, col):", r, c, "map coords:", x, y)
        else:
            r = c = None
    if r is not None:
        with rio.open(PATH_PREDICTIONS) as srcp:
            contig = np.array(
                [srcp.read(i + 1)[r, c] for i in range(len(pred_info["descriptions"]))]
            )
        rows_demo = []
        for j, bk in enumerate(raster_band_keys_ordered):
            ib = pred_info["descriptions"].index(bk)
            score = float(contig[ib])
            tr = th_lookup.loc[bk]
            thr = float(tr["threshold_p10_main"])
            rows_demo.append(
                {
                    "Location": f"r{r}_c{c}",
                    "Species": tr["latin_name"],
                    "Activity": tr["activity_type"],
                    "Continuous score": score,
                    "Threshold (p10)": thr,
                    "Suitable?": score >= thr,
                }
            )
        display(pd.DataFrame(rows_demo))
else:
    print("Skip — aggregate raster missing")


## 10. Conclusions (edit after run)

- **Alignment:** confirm whether `band_key` matched all intended models; review any warnings from §2.
- **Threshold choice:** compare omission rates and `suitable_area_percent` in `threshold_qa_summary.csv`; p10 is a convention, not biology.
- **Problematic models:** rows with `warning` in `threshold_summary.csv` / low `n_presence` / low CV.
- **Sensitivity:** compare the three maps in `figures/potential_species_count_sensitivity_all.png`.
- **Next steps for production:** lift helpers to a module, stream QA without full-raster reads if needed, wire to web visualiser.

**Caveats**

- Maps show *potential* modelled suitability, not confirmed bats.
- Threshold choice directly changes count layers.
- Aggregate counts are model-derived; do not report as field-observed richness.
